# LLM server for parts A and B (Kaggle GPU T4 x2)

Serves the agent model to `00_approach2_remote_partA` and `..._partB` (CPU notebooks) over a public HTTPS
tunnel. Steps: install Ollama, download the model, start a small proxy that (1) requires a secret token and
(2) keeps slow requests alive, then open a Cloudflare quick tunnel to it (no account needed).

**Settings:** Accelerator **GPU T4 x2**, Internet ON. Run it as *Save Version -> Save & Run All* (it keeps
serving for `SERVE_HOURS`), or interactively with the tab open. When section 3 prints the **URL** and
**TOKEN**, put them into the A and B notebooks (Kaggle secrets `LLM_SERVER_URL` and `LLM_SERVER_TOKEN`).
Every restart gives a NEW url and token.


## 1. Model on the local Ollama server


In [ ]:
import os, subprocess, time, json, urllib.request, urllib.error
# ============================ EDIT ONLY IF NEEDED ============================
OLLAMA_MODEL = 'qwen2.5:14b-instruct'   # non-reasoning instruct; alternatives: 'qwen2.5:7b-instruct' (2x faster), 'llama3.1:8b-instruct-q8_0'
NUM_CTX = 32768        # tokens per request; SimuHome agent prompts grow to ~15-30k tokens
PARALLEL = 2           # requests computed at once (each holds a 32k-token cache); others wait in a queue
SERVE_HOURS = 11.5     # Kaggle ends a session at 12 h
# ==============================================================================
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y zstd pciutils lshw >/dev/null 2>&1
if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
    !curl -fsSL https://ollama.com/install.sh | sh
env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NUM_PARALLEL=str(PARALLEL), OLLAMA_MAX_LOADED_MODELS='1',
           OLLAMA_MAX_QUEUE='512', OLLAMA_SCHED_SPREAD='1', OLLAMA_KEEP_ALIVE='24h', OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ollama did not start; see ollama.log')
!ollama pull {OLLAMA_MODEL}
open('Modelfile', 'w').write(f'FROM {OLLAMA_MODEL}\nPARAMETER num_ctx {NUM_CTX}\n')
!ollama create simuhome-agent -f Modelfile
t0 = time.time()
req = urllib.request.Request('http://127.0.0.1:11434/v1/chat/completions', method='POST', headers={'Content-Type': 'application/json'},
    data=json.dumps({'model': 'simuhome-agent', 'temperature': 0, 'messages': [{'role': 'user', 'content': 'Reply with OK'}]}).encode())
print('warm-up reply:', json.loads(urllib.request.urlopen(req, timeout=900).read())['choices'][0]['message']['content'][:40],
      f'({time.time() - t0:.0f} s, includes loading the model onto the GPUs)')
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv


## 2. Proxy: token check + keep-alive

Cloudflare's quick tunnel drops a request that sends nothing for 100 s, and a long prompt can take longer on
a T4. The proxy answers at once with HTTP 200 and sends a space every 15 s until the model's JSON is ready
(leading spaces are valid JSON, so the OpenAI client reads the reply unchanged). Fast replies and errors
(within 15 s) are passed through with their real status code.


In [ ]:
import http.server, secrets, threading
TOKEN = secrets.token_urlsafe(24)
OLLAMA = 'http://127.0.0.1:11434'
STATS = {'requests': 0, 'denied': 0, 'errors': 0, 'busy_seconds': 0.0}

class Proxy(http.server.BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.0'   # the reply ends when the connection closes: no length needed up front
    def log_message(self, *a):
        pass
    def _ok_token(self):
        if self.headers.get('Authorization', '') == f'Bearer {TOKEN}':
            return True
        STATS['denied'] += 1
        self.send_response(401); self.send_header('Content-Type', 'application/json'); self.end_headers()
        self.wfile.write(b'{"error": {"message": "bad token"}}')
        return False
    def _forward(self, method, body=None):
        out = {}
        def work():
            try:
                r = urllib.request.Request(OLLAMA + self.path, data=body, method=method, headers={'Content-Type': 'application/json'})
                with urllib.request.urlopen(r, timeout=6 * 3600) as resp:
                    out['status'], out['body'] = resp.status, resp.read()
            except urllib.error.HTTPError as e:
                out['status'], out['body'] = e.code, e.read()
            except Exception as e:
                out['status'], out['body'] = 502, json.dumps({'error': {'message': repr(e)[:300]}}).encode()
        t0 = time.time(); th = threading.Thread(target=work, daemon=True); th.start(); th.join(15)
        try:
            if not th.is_alive():
                self.send_response(out['status'])
                self.send_header('Content-Type', 'application/json'); self.end_headers()
                self.wfile.write(out['body'])
            else:
                self.send_response(200); self.send_header('Content-Type', 'application/json'); self.end_headers()
                while th.is_alive():
                    self.wfile.write(b' '); self.wfile.flush(); th.join(15)
                self.wfile.write(out['body'])
            if out.get('status', 200) >= 400:
                STATS['errors'] += 1
        except (BrokenPipeError, ConnectionResetError):
            STATS['errors'] += 1   # the client gave up; its request still ran
        STATS['busy_seconds'] += time.time() - t0
    def do_GET(self):
        if self.path == '/health':
            self.send_response(200); self.end_headers(); self.wfile.write(b'ok'); return
        if self._ok_token():
            self._forward('GET')
    def do_POST(self):
        if self._ok_token():
            STATS['requests'] += 1
            self._forward('POST', self.rfile.read(int(self.headers.get('Content-Length', 0))))

server = http.server.ThreadingHTTPServer(('127.0.0.1', 8000), Proxy)
threading.Thread(target=server.serve_forever, daemon=True).start()
print('proxy up on 127.0.0.1:8000')


## 3. Public tunnel: copy the URL and TOKEN printed below


In [ ]:
import re
if not os.path.exists('cloudflared'):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep draining its log
time.sleep(8)   # the address needs a few seconds to become reachable
print('=' * 70)
print('LLM_SERVER_URL   =', URL)
print('LLM_SERVER_TOKEN =', TOKEN)
print('=' * 70)
print('Put both into parts A and B as Kaggle secrets (Add-ons -> Secrets) with exactly these names.')
# self-test through the tunnel, the way parts A and B will call it
req = urllib.request.Request(URL + '/v1/chat/completions', method='POST',
    headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {TOKEN}'},
    data=json.dumps({'model': 'simuhome-agent', 'temperature': 0, 'messages': [{'role': 'user', 'content': 'Reply with OK'}]}).encode())
for attempt in range(6):
    try:
        print('tunnel self-test:', json.loads(urllib.request.urlopen(req, timeout=300).read())['choices'][0]['message']['content'][:40]); break
    except Exception as e:
        print('tunnel not ready yet:', repr(e)[:120]); time.sleep(10)


## 4. Keep serving (this cell runs until `SERVE_HOURS`; stop it to shut the server down)


In [ ]:
T0 = time.time()
while time.time() - T0 < SERVE_HOURS * 3600:
    time.sleep(300)
    if ollama_proc.poll() is not None:
        print('ollama exited; restarting'); ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'a'), stderr=subprocess.STDOUT)
    if tunnel.poll() is not None:
        print('TUNNEL DIED: restart this notebook (it will print a new URL and token)'); break
    print(time.strftime('%H:%M'), f'{(time.time() - T0) / 3600:.1f} h | requests {STATS["requests"]} | errors {STATS["errors"]} | '
          f'denied {STATS["denied"]} | GPU busy ~{STATS["busy_seconds"] / max(1, time.time() - T0):.1f} requests in flight on average', flush=True)
